# Temperature upper-slack sensitivity

This notebook reproduces the observed-future MIQP sensitivity study at exactly
`w_x_slack = 100, 1000, 10000`. It uses the same fixed baselines, thermal
surrogates, PV coefficients, and 22-date study split as the main study.


The measured L14 table is private and is not distributed. Supply it at
`data/private/l14_merged_data_with_rain.csv`. Outputs are written to
`outputs/simulations/weight_sweep/`. Gurobi is discovered normally from the
active Python environment.


In [ ]:
import json
import os
import platform
import sys
from collections import OrderedDict
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "mmv4ef").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the MMV4EF release repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mmv4ef.mpc_future_data import apply_future_data_source, build_forecast_audit
from mmv4ef.temperature_bound_sweep import (
    DenominatorAwareRatioContract,
    SWEEP_WEIGHTS as _LIBRARY_SWEEP_WEIGHTS,
    build_balanced_selection,
    find_nonfinite_numeric_failures,
    format_validation_failures,
    optimize_miqp_with_numeric_retry,
    temperature_metrics,
    validate_denominator_aware_ratios,
)

SWEEP_WEIGHTS = (100.0, 1000.0, 10000.0)
if tuple(_LIBRARY_SWEEP_WEIGHTS) != SWEEP_WEIGHTS:
    raise RuntimeError("Notebook and library sweep weights do not match.")

L14_DATA_PATH = Path(os.environ.get("MMV4EF_DATA", REPO_ROOT / "data/private/l14_merged_data_with_rain.csv"))
AC_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "ac_model.pth"
NV_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "nv_model.pth"
PV_COEFFICIENT_PATH = REPO_ROOT / "models" / "pv" / "pv_appendix_b_best_model.csv"
OUTPUT_DIR = Path(os.environ.get("MMV4EF_OUTPUT_DIR", REPO_ROOT / "outputs/simulations/weight_sweep"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
# Shared final-study implementation (no duplicated controller).
from mmv4ef.controller import *
torch.set_num_threads(1)


In [ ]:
# Shared final-study implementation (no duplicated controller).


In [ ]:
# Shared final-study implementation (no duplicated controller).


In [ ]:
# Shared final-study implementation (no duplicated controller).


## Fixed configuration and three-weight grid


In [ ]:
segment_split = 'test+val'
test_date_policy = 'union'
max_segments_env = os.environ.get('MMV4EF_MAX_SEGMENTS', '').strip()
max_segments_to_run = int(max_segments_env) if max_segments_env else None
if max_segments_to_run is not None and max_segments_to_run < 1:
    raise ValueError(
        'MMV4EF_MAX_SEGMENTS must be a positive integer when set.'
    )

window_mode = os.environ.get('MMV4EF_WINDOW', 'full').strip().lower()
if window_mode == 'smoke':
    run_window = dict(start_hhmm='07:30', end_hhmm='09:00')
    mpc_overrides = dict(horizon_min=60, mip_time_limit=600)
elif window_mode == 'full':
    run_window = dict(start_hhmm='07:30', end_hhmm='19:00')
    mpc_overrides = dict(horizon_min=60)
else:
    raise ValueError("MMV4EF_WINDOW must be 'full' or 'smoke'.")

save_timeseries = True
show_daily_figures = False

pv_source_area_sqft = 5000.0
building_area_sqft = 75.0
pv_scale_factor = building_area_sqft / pv_source_area_sqft

best_model = pd.read_csv(PV_COEFFICIENT_PATH).iloc[0]
coeffs = {
    'a1': float(best_model['a1']),
    'a2': float(best_model['a2']),
    'a3': float(best_model['a3']),
}

SIMPLE_WEIGHTS = dict(
    energy_weight=1.667e3,
    smoothness_weight=1e-1,
    pv_self_consumption_weight=20.0,
)

FIXED_MPC_KWARGS = dict(
    T_ref_ac=27.0,
    T_nv_max=30.5,
    T_nv_soft=30.0,
    x_min=22.0,
    x_max=30.0,
    pmv_target=-0.5,
    pmv_model='linear',
    rh_ac_fixed=65.0,
    rh_nv_source_col='OutdoorHumidityWindow',
    pmv_big_m=5.0,
    energy_norm_kwh=10.0,
    cop_fcu=5.0,
    cop_pfc=5.0,
    T_out_nv_ref=30.0,
    min_dwell_steps=12,
    enforce_current_rain_safety=True,
    rain_active_threshold=0.5,
    rain_lockout_min=30,
    ignore_rain_in_mpc_prediction=True,
    linearize_energy=True,
    w_terminal=2e1,
    w_switch=1e1,
    w_x_slack=SWEEP_WEIGHTS[0],
)

FCU_SUPPLY_COLS = [
    'fcu01_supply', 'fcu02_supply', 'fcu03_supply', 'fcu04_supply',
    'fcu05_supply',
]
PFC_SUPPLY_COLS = ['pfc01_supply', 'pfc02_supply']




base_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=False, **mpc_overrides
)
pv_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides
)
pv_metric_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides
)

BASELINE_CASES = ('AC baseline', 'RBC baseline')
MIQP_CASE_SPECS = tuple(
    spec
    for weight in SWEEP_WEIGHTS
    for spec in (
        (
            f'MIQP no PV | observed future | wx={int(weight)}',
            replace(base_cfg, w_x_slack=weight),
            'no_pv',
            'observed',
            weight,
        ),
        (
            f'MIQP onsite PV | observed future | wx={int(weight)}',
            replace(pv_cfg, w_x_slack=weight),
            'onsite_pv',
            'observed',
            weight,
        ),
    )
)
CASE_ORDER = [*BASELINE_CASES, *(spec[0] for spec in MIQP_CASE_SPECS)]
CASE_METADATA = {
    'AC baseline': {
        'controller_objective': 'baseline_ac',
        'forecast_source': 'not_applicable',
        'w_x_slack': np.nan,
    },
    'RBC baseline': {
        'controller_objective': 'baseline_rbc',
        'forecast_source': 'not_applicable',
        'w_x_slack': np.nan,
    },
    **{
        name: {
            'controller_objective': objective,
            'forecast_source': forecast_source,
            'w_x_slack': float(weight),
        }
        for name, _, objective, forecast_source, weight in MIQP_CASE_SPECS
    },
}

hooks = make_default_hooks(
    data_path=str(L14_DATA_PATH),
    ac_model_path=AC_MODEL_PATH,
    nv_model_path=NV_MODEL_PATH,
    test_date_policy=test_date_policy,
    segment_split=segment_split,
    min_seg_len=600,
)
segments = hooks._segments_cached()
if not segments:
    raise ValueError('No clean segments are available for the requested split/policy.')

if max_segments_to_run is None:
    selected_indices = list(range(len(segments)))
else:
    selected_indices = list(range(min(max_segments_to_run, len(segments))))
selected_dates = [
    segments[index]['date'].iloc[0].date().isoformat()
    for index in selected_indices
]
run_suffix = 'all' if max_segments_to_run is None else f'first{len(selected_indices)}'
STUDY_SLUG = 'control_case_comparison_w_x_slack_sweep'
# The run directory already carries the collision-proof run ID. Keep artifact
# names short enough for the nested Windows worktree path.
run_stem = 'wx_sweep'

print({
    'output_dir': OUTPUT_DIR.name,
    'selected_dates': selected_dates,
    'window_mode': window_mode,
    'run_window': run_window,
    'horizon_min': base_cfg.horizon_min,
    'sweep_weights': SWEEP_WEIGHTS,
    'case_count': len(CASE_ORDER),
    'study_slug': STUDY_SLUG,
    'gurobi_available': GUROBI_OK,
})

## Fixed-policy baselines


In [ ]:
# Shared final-study implementation (no duplicated controller).


## Metrics, solver provenance, and validation helpers


In [ ]:
def enrich_results(
    sim_df: pd.DataFrame,
    cfg: MPCConfig,
    case_name: Optional[str] = None,
    segment_date: Optional[str] = None,
    forecast_source: Optional[str] = None,
    controller_objective: Optional[str] = None,
    w_x_slack: Optional[float] = None,
) -> pd.DataFrame:
    out = sim_df.copy()
    metadata = CASE_METADATA.get(case_name, {})
    if case_name is not None:
        out['case'] = case_name
    if segment_date is not None:
        out['date'] = segment_date
    out['forecast_source'] = (
        forecast_source
        if forecast_source is not None
        else metadata.get('forecast_source', 'not_applicable')
    )
    out['controller_objective'] = (
        controller_objective
        if controller_objective is not None
        else metadata.get('controller_objective', 'not_applicable')
    )
    out['w_x_slack'] = (
        float(w_x_slack)
        if w_x_slack is not None
        else metadata.get('w_x_slack', np.nan)
    )
    out['pv_kw'] = [
        pv_available_power_kw(cfg, I_solar=i_solar, T_out=t_out)
        for i_solar, t_out in zip(out['I_solar'], out['T_out'])
    ]
    out['hvac_kw'] = [
        hvac_power_heat_exchange_kw(
            cfg,
            z_mode=row['z'],
            zone_temps=row[AC_STATE_COLS].to_numpy(dtype=float),
            fcu_supplies=row[FCU_SUPPLY_COLS]
            .fillna(row['T_out'])
            .to_numpy(dtype=float),
            pfc_supplies=row[PFC_SUPPLY_COLS].to_numpy(dtype=float),
            T_out=row['T_out'],
        )
        for _, row in out.iterrows()
    ]
    out['grid_kw'] = np.maximum(out['hvac_kw'] - out['pv_kw'], 0.0)
    out['export_kw'] = np.maximum(out['pv_kw'] - out['hvac_kw'], 0.0)
    out['self_kw'] = np.minimum(out['hvac_kw'], out['pv_kw'])
    return out


def peak_window_metrics(df: pd.DataFrame) -> dict:
    timestamps = pd.DatetimeIndex(
        pd.to_datetime(df['ts'] if 'ts' in df.columns else df.index)
    )
    if timestamps.isna().any():
        raise ValueError('Peak-window metrics require finite timestamps.')
    minutes = timestamps.hour * 60 + timestamps.minute
    # Match time_in_windows exactly: 07:30-10:00 and 16:30-19:00.
    morning = (minutes >= 7 * 60 + 30) & (minutes < 10 * 60)
    evening = (minutes >= 16 * 60 + 30) & (minutes < 19 * 60)
    configured_union = np.asarray(
        [time_in_windows(timestamp) for timestamp in timestamps], dtype=bool
    )
    if not np.array_equal(morning | evening, configured_union):
        raise RuntimeError('Peak-window masks drifted from time_in_windows.')

    def summarize(mask, prefix: str) -> dict:
        represented_dates = pd.Index(timestamps.normalize().unique())
        window_timestamps = timestamps[mask].floor('min')
        observed_date_minutes = pd.DataFrame({
            'date': window_timestamps.normalize(),
            'minute': window_timestamps.hour * 60 + window_timestamps.minute,
        }).drop_duplicates()
        observed_by_date = (
            observed_date_minutes.groupby('date').size()
            .reindex(represented_dates, fill_value=0)
        )
        minutes_per_complete_window = 150
        available = bool(
            len(represented_dates) > 0
            and observed_by_date.eq(minutes_per_complete_window).all()
        )
        return {
            f'{prefix}_peak_available': int(available),
            f'{prefix}_peak_observed_minutes': int(observed_by_date.sum()),
            f'{prefix}_peak_expected_minutes': int(
                len(represented_dates) * minutes_per_complete_window
            ),
            f'{prefix}_peak_hvac_kw': (
                float(df.loc[mask, 'hvac_kw'].max())
                if available else np.nan
            ),
            f'{prefix}_peak_grid_kw': (
                float(df.loc[mask, 'grid_kw'].max())
                if available else np.nan
            ),
        }

    return {
        **summarize(morning, 'morning'),
        **summarize(evening, 'evening'),
    }


def summarize_case(
    df: pd.DataFrame,
    pv_window_threshold_kw: Optional[float] = None,
) -> dict:
    step_h = 1.0 / 60.0
    if pv_window_threshold_kw is None:
        pv_window_threshold_kw = max(float(df['pv_kw'].max()) * 0.05, 1e-9)
    pv_window = df['pv_kw'] > pv_window_threshold_kw
    hvac_sum = float(df['hvac_kw'].sum())
    pv_sum = float(df['pv_kw'].sum())
    self_sum = float(df['self_kw'].sum())
    window_open = 1 - df['z'].astype(int)
    thermal = temperature_metrics(df, AC_STATE_COLS, bound_c=30.0)
    post5_thermal = {
        f'post5_{name}': value
        for name, value in temperature_metrics(
            df,
            AC_STATE_COLS,
            bound_c=30.0,
            exclude_first_minutes=5,
        ).items()
    }
    below = np.maximum(22.0 - df[AC_STATE_COLS].to_numpy(dtype=float), 0.0)
    rain_force = pd.to_numeric(
        df.get('rain_force_ac', pd.Series(0, index=df.index)),
        errors='coerce',
    ).fillna(0).astype(int)
    rain_lock = pd.to_numeric(
        df.get('rain_lock_steps_before', pd.Series(0, index=df.index)),
        errors='coerce',
    ).fillna(0).astype(int)
    rain_safety = ((rain_force > 0) | (rain_lock > 0)) & (
        df['z'].astype(int) != 1
    )
    peak_windows = peak_window_metrics(df)
    return {
        'hvac_kwh': hvac_sum * step_h,
        'pv_available_kwh': pv_sum * step_h,
        'grid_kwh': float(df['grid_kw'].sum()) * step_h,
        'export_kwh': float(df['export_kw'].sum()) * step_h,
        'self_consumed_kwh': self_sum * step_h,
        'self_consumption_ratio': self_sum / pv_sum if pv_sum > 0 else np.nan,
        'self_sufficiency_ratio': self_sum / hvac_sum if hvac_sum > 0 else np.nan,
        'pv_window_hvac_kwh': float(df.loc[pv_window, 'hvac_kw'].sum()) * step_h,
        'non_pv_window_hvac_kwh': (
            float(df.loc[~pv_window, 'hvac_kw'].sum()) * step_h
        ),
        'pv_window_hvac_share_pct': (
            100.0 * float(df.loc[pv_window, 'hvac_kw'].sum()) / hvac_sum
            if hvac_sum > 0 else np.nan
        ),
        'peak_hvac_kw': float(df['hvac_kw'].max()),
        'peak_grid_kw': float(df['grid_kw'].max()),
        'mean_temp_c': float(df['T_mean'].mean()),
        'max_temp_c': float(df['T_mean'].max()),
        'min_temp_c': float(df['T_mean'].min()),
        'zone_min_c': float(df[AC_STATE_COLS].to_numpy(dtype=float).min()),
        'zone_minutes_below_22': int(np.any(below > 0.0, axis=1).sum()),
        'max_below_22_c': float(below.max()),
        'ac_fraction': float(df['z'].mean()),
        'pfc_on_fraction': float(df['pfc_on'].astype(int).mean()),
        'window_open_fraction': float(window_open.mean()),
        'window_open_minutes': int(window_open.sum()),
        'switch_count': int(
            (df['z'].astype(int).diff().abs().fillna(0) > 0).sum()
        ),
        'rain_forced_ac_minutes': int(rain_force.sum()),
        'rain_lockout_minutes': int((rain_lock > 0).sum()),
        'rain_safety_violation_minutes': int(rain_safety.sum()),
        **thermal,
        **post5_thermal,
        **peak_windows,
    }


def build_metric_table(case_results: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    max_pv = max(float(df['pv_kw'].max()) for df in case_results.values())
    threshold = max(max_pv * 0.05, 1e-9)
    metrics = pd.DataFrame.from_dict(
        {
            name: summarize_case(df, pv_window_threshold_kw=threshold)
            for name, df in case_results.items()
        },
        orient='index',
    ).loc[CASE_ORDER]
    metrics['controller_objective'] = [
        CASE_METADATA[name]['controller_objective'] for name in metrics.index
    ]
    metrics['forecast_source'] = [
        CASE_METADATA[name]['forecast_source'] for name in metrics.index
    ]
    metrics['w_x_slack'] = [
        CASE_METADATA[name]['w_x_slack'] for name in metrics.index
    ]
    ac_hvac = float(metrics.loc['AC baseline', 'hvac_kwh'])
    rbc_hvac = float(metrics.loc['RBC baseline', 'hvac_kwh'])
    metrics['energy_savings_vs_ac_pct'] = (
        100.0 * (ac_hvac - metrics['hvac_kwh']) / ac_hvac
        if ac_hvac > 0.05 else np.nan
    )
    metrics['energy_shedding_vs_rbc_kwh'] = rbc_hvac - metrics['hvac_kwh']
    metrics['energy_shedding_vs_rbc_pct'] = (
        100.0 * (rbc_hvac - metrics['hvac_kwh']) / rbc_hvac
        if rbc_hvac > 0.05 else np.nan
    )
    metrics['energy_savings_vs_rbc_pct'] = metrics[
        'energy_shedding_vs_rbc_pct'
    ]
    metrics['grid_reduction_vs_miqp_no_pv_kwh'] = np.nan
    metrics['load_shifted_to_pv_window_kwh'] = np.nan
    for weight in SWEEP_WEIGHTS:
        no_pv_name = f'MIQP no PV | observed future | wx={int(weight)}'
        pv_name = f'MIQP onsite PV | observed future | wx={int(weight)}'
        reference = metrics.loc[no_pv_name]
        metrics.loc[
            [no_pv_name, pv_name], 'grid_reduction_vs_miqp_no_pv_kwh'
        ] = reference['grid_kwh'] - metrics.loc[
            [no_pv_name, pv_name], 'grid_kwh'
        ]
        metrics.loc[
            [no_pv_name, pv_name], 'load_shifted_to_pv_window_kwh'
        ] = metrics.loc[
            [no_pv_name, pv_name], 'pv_window_hvac_kwh'
        ] - reference['pv_window_hvac_kwh']
    return metrics


DECISION_SUMMARY_COLUMNS = (
    'decision_count',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count',
    'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s',
    'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s',
    'solver_runtime_sum_s',
    'solver_runtime_mean_s',
    'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
    'solver_mip_gap_mean',
    'solver_mip_gap_max',
)


def summarize_decision_diagnostics(frame: pd.DataFrame) -> dict:
    if frame.empty:
        return {
            'decision_count': 0,
            'predicted_upper_slack_sum': np.nan,
            'predicted_upper_slack_positive_fraction': np.nan,
            'numeric_retry_count': 0,
            'numeric_retry_fraction': np.nan,
            'inf_or_unbd_disambiguation_count': 0,
            'inf_or_unbd_disambiguation_fraction': np.nan,
            'solver_initial_runtime_sum_s': np.nan,
            'solver_retry_runtime_sum_s': np.nan,
            'solver_inf_or_unbd_disambiguation_runtime_sum_s': np.nan,
            'solver_total_runtime_sum_s': np.nan,
            'solver_runtime_sum_s': np.nan,
            'solver_runtime_mean_s': np.nan,
            'solver_runtime_max_s': np.nan,
            'solver_mip_gap_available_fraction': np.nan,
            'solver_mip_gap_mean': np.nan,
            'solver_mip_gap_max': np.nan,
        }
    gap_available = pd.to_numeric(
        frame['solver_mip_gap_available'], errors='coerce'
    )
    gap_values = pd.to_numeric(frame['solver_mip_gap'], errors='coerce')
    available_gaps = gap_values.loc[gap_available == 1]
    runtime = pd.to_numeric(frame['solver_runtime_s'], errors='coerce')
    retry_used = pd.to_numeric(frame['numeric_retry_used'], errors='coerce')
    inf_or_unbd_disambiguation_used = pd.to_numeric(
        frame['inf_or_unbd_disambiguation_used'], errors='coerce'
    )
    initial_runtime = pd.to_numeric(
        frame['solver_initial_runtime_s'], errors='coerce'
    )
    retry_runtime = pd.to_numeric(
        frame['solver_retry_runtime_s'], errors='coerce'
    )
    inf_or_unbd_disambiguation_runtime = pd.to_numeric(
        frame['solver_inf_or_unbd_disambiguation_runtime_s'], errors='coerce'
    )
    total_runtime = pd.to_numeric(
        frame['solver_total_runtime_s'], errors='coerce'
    )
    return {
        'decision_count': int(len(frame)),
        'predicted_upper_slack_sum': float(
            pd.to_numeric(
                frame['predicted_upper_slack_sum'], errors='coerce'
            ).sum()
        ),
        'predicted_upper_slack_positive_fraction': float(
            pd.to_numeric(
                frame['predicted_upper_slack_positive'], errors='coerce'
            ).mean()
        ),
        'numeric_retry_count': int(retry_used.sum()),
        'numeric_retry_fraction': float(retry_used.mean()),
        'inf_or_unbd_disambiguation_count': int(
            inf_or_unbd_disambiguation_used.sum()
        ),
        'inf_or_unbd_disambiguation_fraction': float(
            inf_or_unbd_disambiguation_used.mean()
        ),
        'solver_initial_runtime_sum_s': float(initial_runtime.sum()),
        'solver_retry_runtime_sum_s': float(retry_runtime.sum()),
        'solver_inf_or_unbd_disambiguation_runtime_sum_s': float(
            inf_or_unbd_disambiguation_runtime.sum()
        ),
        'solver_total_runtime_sum_s': float(total_runtime.sum()),
        'solver_runtime_sum_s': float(runtime.sum()),
        'solver_runtime_mean_s': float(runtime.mean()),
        'solver_runtime_max_s': float(runtime.max()),
        'solver_mip_gap_available_fraction': float(gap_available.mean()),
        'solver_mip_gap_mean': (
            float(available_gaps.mean()) if len(available_gaps) else np.nan
        ),
        'solver_mip_gap_max': (
            float(available_gaps.max()) if len(available_gaps) else np.nan
        ),
    }


def validate_solver_recovery_provenance(frame, grb):
    retries = pd.to_numeric(frame['solver_retry_count'])
    flags = pd.to_numeric(frame['numeric_retry_used'])
    final = pd.to_numeric(frame['solver_final_status'])
    initial_runtime = pd.to_numeric(frame['solver_initial_runtime_s'])
    retry_runtime = pd.to_numeric(frame['solver_retry_runtime_s'])
    total_runtime = pd.to_numeric(frame['solver_total_runtime_s'])
    return {
        'provenance_valid': bool((retries >= 0).all() and flags.eq(retries.gt(0).astype(int)).all()
                                 and final.eq(grb.OPTIMAL).all()
                                 and pd.to_numeric(frame['solver_status']).eq(final).all()),
        'runtime_accounting_valid': bool((initial_runtime >= 0).all() and (retry_runtime >= 0).all()
                                         and (total_runtime >= initial_runtime + retry_runtime - 1e-9).all()),
    }


def attach_decision_diagnostic_metrics(
    metrics: pd.DataFrame,
    decision_records: list[dict],
) -> pd.DataFrame:
    enriched = metrics.copy()
    for column in DECISION_SUMMARY_COLUMNS:
        enriched[column] = 0 if column == 'decision_count' else np.nan
    diagnostics = pd.DataFrame(decision_records)
    if diagnostics.empty:
        return enriched
    for case_name, case_diagnostics in diagnostics.groupby('case', sort=False):
        if case_name not in enriched.index:
            raise ValueError(f'Unknown diagnostic case: {case_name}')
        summary = summarize_decision_diagnostics(case_diagnostics)
        for column, value in summary.items():
            enriched.loc[case_name, column] = value
    return enriched


def metric_table_to_records(metrics: pd.DataFrame, segment_date: str) -> pd.DataFrame:
    records = metrics.reset_index().rename(columns={'index': 'case'})
    records.insert(0, 'date', segment_date)
    return records


def sparse_progress(case_name: str, segment_date: str):
    def report(step_idx: int, total_steps: int, timestamp: pd.Timestamp) -> None:
        if step_idx == 0 or step_idx + 1 == total_steps or step_idx % 24 == 0:
            print(
                f'  {segment_date} | {case_name} | '
                f'{step_idx + 1}/{total_steps} @ {timestamp}'
            )
    return report


def _write_csv_exclusive(
    frame: pd.DataFrame,
    path: Path,
    *,
    compression: Optional[str] = None,
) -> None:
    if path.exists():
        raise FileExistsError(f'Refusing existing artifact: {path}')
    frame.to_csv(path, index=False, mode='x', compression=compression)


def _savefig_exclusive(fig, path: Path, *, dpi: int = 180) -> None:
    if path.exists():
        raise FileExistsError(f'Refusing existing figure: {path}')
    with path.open('xb') as handle:
        fig.savefig(handle, format='png', dpi=dpi, bbox_inches='tight')




def _finite_numeric(frame: pd.DataFrame, columns: list[str]) -> bool:
    if any(column not in frame.columns for column in columns):
        return False
    values = frame.loc[:, columns].apply(pd.to_numeric, errors='coerce')
    return bool(np.isfinite(values.to_numpy(dtype=float)).all())


def _validation_row(name: str, expected, actual, passed: bool) -> dict:
    return {
        'check': name,
        'expected': expected,
        'actual': actual,
        'passed': bool(passed),
    }

## Run the sweep


In [ ]:
artifact_paths = {
    'timeseries': OUTPUT_DIR / f'{run_stem}_timeseries.csv.gz',
    'decision_diagnostics': (
        OUTPUT_DIR / f'{run_stem}_decision_diagnostics.csv.gz'
    ),
    'daily_metrics': OUTPUT_DIR / f'{run_stem}_daily_metrics.csv',
    'pooled_metrics': OUTPUT_DIR / f'{run_stem}_pooled_metrics.csv',
    'pareto_table': OUTPUT_DIR / f'{run_stem}_pareto_table.csv',
    'validation': OUTPUT_DIR / f'{run_stem}_validation.csv',
    'errors': OUTPUT_DIR / f'{run_stem}_errors.csv',
    'run_metadata': OUTPUT_DIR / f'{run_stem}_run_metadata.json',
    'weight_response': OUTPUT_DIR / f'{run_stem}_weight_response.png',
    'pareto': OUTPUT_DIR / f'{run_stem}_pareto.png',
    'representative_days': OUTPUT_DIR / f'{run_stem}_representative_days.csv',
}
representative_dir = OUTPUT_DIR / f'{run_stem}_representative_days'
existing_artifacts = [
    str(path) for path in [*artifact_paths.values(), representative_dir]
    if path.exists()
]
if existing_artifacts:
    raise FileExistsError(
        f'Refusing existing intended artifacts: {existing_artifacts}'
    )

if not GUROBI_OK:
    raise RuntimeError('Gurobi is not available; MIQP cases cannot run.')

all_case_results = {case: [] for case in CASE_ORDER}
daily_summary_records = []
timeseries_records = []
error_records = []
decision_diagnostic_records = []

for run_pos, local_idx in enumerate(selected_indices, start=1):
    segment_date = segments[local_idx]['date'].iloc[0].date().isoformat()
    print(f'Running segment {run_pos}/{len(selected_indices)}: {segment_date}')
    diagnostic_record_start = len(decision_diagnostic_records)
    active_case_name = None
    active_controller_objective = None
    active_w_x_slack = np.nan
    try:
        case_results = {}
        active_case_name = 'AC baseline'
        active_controller_objective = 'baseline_ac'
        ac_df = simulate_policy_day(
            'AC baseline', hooks, pv_metric_cfg, day_idx=local_idx,
            policy='ac', **run_window,
        )
        case_results['AC baseline'] = enrich_results(
            ac_df, pv_metric_cfg, 'AC baseline', segment_date,
        )
        active_case_name = 'RBC baseline'
        active_controller_objective = 'baseline_rbc'
        rbc_df = simulate_policy_day(
            'RBC baseline', hooks, pv_metric_cfg, day_idx=local_idx,
            policy='rbc', **run_window,
        )
        case_results['RBC baseline'] = enrich_results(
            rbc_df, pv_metric_cfg, 'RBC baseline', segment_date,
        )

        for (
            case_name,
            case_cfg,
            controller_objective,
            forecast_source,
            weight,
        ) in MIQP_CASE_SPECS:
            active_case_name = case_name
            active_controller_objective = controller_objective
            active_w_x_slack = float(weight)
            hooks.reset_cache()
            miqp_df = simulate_day(
                day_idx=local_idx,
                hooks=hooks,
                cfg=case_cfg,
                init_state_cols=hooks.state_cols,
                progress_fn=sparse_progress(case_name, segment_date),
                forecast_source='observed',
                forecast_case_name=case_name,
                forecast_controller_objective=controller_objective,
                decision_diagnostic_records=decision_diagnostic_records,
                **run_window,
            )
            case_results[case_name] = enrich_results(
                miqp_df,
                pv_metric_cfg,
                case_name,
                segment_date,
                forecast_source=forecast_source,
                controller_objective=controller_objective,
                w_x_slack=weight,
            )

        active_case_name = None
        active_controller_objective = None
        active_w_x_slack = np.nan
        daily_metrics = attach_decision_diagnostic_metrics(
            build_metric_table(case_results),
            decision_diagnostic_records[diagnostic_record_start:],
        )
        daily_summary_records.append(
            metric_table_to_records(daily_metrics, segment_date)
        )
        for case_name, frame in case_results.items():
            all_case_results[case_name].append(frame)
            timeseries_records.append(frame.reset_index())
    except Exception as exc:
        error_records.append({
            'date': segment_date,
            'segment_index': local_idx,
            'case': active_case_name,
            'controller_objective': active_controller_objective,
            'w_x_slack': active_w_x_slack,
            'error_type': type(exc).__name__,
            'error': repr(exc),
        })
        print(f'ERROR on {segment_date}: {exc!r}')

errors_df = pd.DataFrame(
    error_records,
    columns=[
        'date', 'segment_index', 'case', 'controller_objective', 'w_x_slack',
        'error_type', 'error',
    ],
)
if len(errors_df):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    failure_validation = pd.DataFrame([
        _validation_row('no_error_rows', 0, len(errors_df), False)
    ])
    _write_csv_exclusive(failure_validation, artifact_paths['validation'])
    raise RuntimeError(
        f"Sweep recorded {len(errors_df)} date/solver errors; "
        f"see {artifact_paths['errors']}"
    )

if not daily_summary_records or not timeseries_records:
    raise RuntimeError('No complete sweep rows were produced.')

missing_cases = [
    case for case in CASE_ORDER if not all_case_results.get(case)
]
if missing_cases:
    raise RuntimeError(f'Missing completed results for cases: {missing_cases}')

daily_metrics_df = pd.concat(daily_summary_records, ignore_index=True)
all_timeseries_df = pd.concat(timeseries_records, ignore_index=True)
decision_diagnostics_df = pd.DataFrame(decision_diagnostic_records)
expected_case_count = len(CASE_ORDER)
expected_miqp_case_count = len(MIQP_CASE_SPECS)

if window_mode == 'full':
    expected_date_count = 22
    expected_daily_rows = 22 * expected_case_count
    expected_timeseries_rows = 22 * expected_case_count * 690
    expected_miqp_decisions = 22 * expected_miqp_case_count * 138
    expected_minutes_per_case_date = 690
    expected_decisions_per_miqp_case_date = 138
else:
    expected_date_count = 1
    expected_daily_rows = expected_case_count
    expected_timeseries_rows = expected_case_count * 90
    expected_miqp_decisions = expected_miqp_case_count * 18
    expected_minutes_per_case_date = 90
    expected_decisions_per_miqp_case_date = 18


## Validate and pool results


In [ ]:
required_timeseries_columns = {
    'ts', 'date', 'case', 'controller_objective', 'forecast_source',
    'w_x_slack', 'z', 'pfc_on', 'T_mean', 'T_out', 'I_solar',
    'hvac_kw', 'pv_kw', 'grid_kw', 'export_kw', 'self_kw', *AC_STATE_COLS,
}
required_daily_columns = {
    'date', 'case', 'controller_objective', 'w_x_slack',
    'zone_degree_minutes_above_30', 'post5_zone_degree_minutes_above_30',
    'any_zone_minutes_above_30', 'post5_any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'pv_available_kwh', 'grid_kwh',
    'export_kwh', 'self_consumed_kwh', 'self_consumption_ratio',
    'self_sufficiency_ratio', 'peak_hvac_kw', 'peak_grid_kw',
    'morning_peak_available', 'morning_peak_observed_minutes',
    'morning_peak_expected_minutes', 'morning_peak_hvac_kw',
    'morning_peak_grid_kw', 'evening_peak_available',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'evening_peak_hvac_kw', 'evening_peak_grid_kw',
    'energy_shedding_vs_rbc_kwh', 'energy_shedding_vs_rbc_pct',
    'pfc_on_fraction', 'window_open_fraction', 'switch_count',
    'rain_safety_violation_minutes', 'decision_count',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s',
    'solver_runtime_sum_s', 'solver_runtime_mean_s',
    'solver_runtime_max_s', 'solver_mip_gap_available_fraction',
    'solver_mip_gap_mean', 'solver_mip_gap_max',
}
required_decision_columns = {
    'date', 'case', 'decision_time', 'controller_objective',
    'forecast_source', 'w_x_slack', 'chosen_mode_z', 'previous_mode_z',
    'pfcu_on', 'dwell_lock_steps_before', 'effective_dwell_lock_steps',
    'rain_observed_active', 'rain_force_ac', 'rain_lock_steps_before',
    'rain_lock_steps_after', 'objective_total', 'objective_comfort',
    'objective_du', 'objective_switch', 'objective_energy',
    'objective_x_slack', 'objective_pv', 'predicted_max_temp_c',
    'predicted_upper_slack_sum', 'predicted_upper_slack_positive',
    'solver_initial_status', 'solver_final_status', 'solver_status',
    'numeric_retry_used', 'solver_retry_count',
    'inf_or_unbd_disambiguation_used',
    'solver_inf_or_unbd_disambiguation_count',
    'solver_initial_runtime_s', 'solver_retry_runtime_s',
    'solver_inf_or_unbd_disambiguation_runtime_s',
    'solver_total_runtime_s', 'solver_retry_numeric_focus',
    'solver_runtime_s', 'solver_mip_gap', 'solver_mip_gap_available',
}
missing_required = sorted(
    (required_timeseries_columns - set(all_timeseries_df.columns))
    | (required_daily_columns - set(daily_metrics_df.columns))
    | (required_decision_columns - set(decision_diagnostics_df.columns))
)

timeseries_numeric = [
    *AC_STATE_COLS, 'z', 'pfc_on', 'T_mean', 'T_out', 'I_solar',
    'hvac_kw', 'pv_kw', 'grid_kw', 'export_kw', 'self_kw',
]
daily_numeric = [
    'zone_degree_minutes_above_30', 'post5_zone_degree_minutes_above_30',
    'any_zone_minutes_above_30', 'post5_any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'pv_available_kwh', 'grid_kwh',
    'export_kwh', 'self_consumed_kwh', 'peak_hvac_kw', 'peak_grid_kw',
    'morning_peak_available', 'morning_peak_observed_minutes',
    'morning_peak_expected_minutes', 'evening_peak_available',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'energy_shedding_vs_rbc_kwh',
    'pfc_on_fraction', 'window_open_fraction', 'switch_count',
    'rain_safety_violation_minutes',
]
decision_numeric = [
    'w_x_slack', 'chosen_mode_z', 'previous_mode_z', 'pfcu_on',
    'dwell_lock_steps_before', 'effective_dwell_lock_steps',
    'rain_observed_active', 'rain_force_ac', 'rain_lock_steps_before',
    'rain_lock_steps_after', 'objective_total', 'objective_comfort',
    'objective_du', 'objective_switch', 'objective_energy',
    'objective_x_slack', 'objective_pv', 'predicted_max_temp_c',
    'predicted_upper_slack_sum', 'predicted_upper_slack_positive',
    'solver_initial_status', 'solver_final_status', 'solver_status',
    'numeric_retry_used', 'solver_retry_count',
    'inf_or_unbd_disambiguation_used',
    'solver_inf_or_unbd_disambiguation_count',
    'solver_initial_runtime_s', 'solver_retry_runtime_s',
    'solver_inf_or_unbd_disambiguation_runtime_s',
    'solver_total_runtime_s', 'solver_runtime_s',
    'solver_mip_gap_available',
]
nonfinite_failures = pd.concat(
    [
        find_nonfinite_numeric_failures(
            all_timeseries_df,
            timeseries_numeric,
            table_name='timeseries',
        ),
        find_nonfinite_numeric_failures(
            daily_metrics_df,
            daily_numeric,
            table_name='daily_metrics',
        ),
        find_nonfinite_numeric_failures(
            decision_diagnostics_df,
            decision_numeric,
            table_name='decision_diagnostics',
        ),
    ],
    ignore_index=True,
)
numeric_finite = not missing_required and nonfinite_failures.empty
numeric_finite_actual = (
    True
    if numeric_finite
    else format_validation_failures(nonfinite_failures)
)

daily_ratio_frame = daily_metrics_df.copy()
daily_ratio_base_columns = {'date', 'controller_objective', 'hvac_kwh'}
if daily_ratio_base_columns.issubset(daily_ratio_frame.columns):
    daily_rbc_hvac_by_date = (
        daily_ratio_frame.loc[
            daily_ratio_frame['controller_objective'] == 'baseline_rbc'
        ]
        .groupby('date', sort=False)['hvac_kwh']
        .first()
    )
    daily_ratio_frame['_daily_rbc_hvac_kwh'] = daily_ratio_frame['date'].map(
        daily_rbc_hvac_by_date
    )
    daily_ratio_frame['_daily_energy_shedding_vs_rbc_kwh'] = (
        daily_ratio_frame['_daily_rbc_hvac_kwh']
        - daily_ratio_frame['hvac_kwh']
    )
daily_ratio_contracts = [
    DenominatorAwareRatioContract(
        'self_consumption_ratio',
        'self_consumed_kwh',
        'pv_available_kwh',
        denominator_guard_gt=0.0,
    ),
    DenominatorAwareRatioContract(
        'self_sufficiency_ratio',
        'self_consumed_kwh',
        'hvac_kwh',
        denominator_guard_gt=0.0,
    ),
    DenominatorAwareRatioContract(
        'energy_shedding_vs_rbc_pct',
        '_daily_energy_shedding_vs_rbc_kwh',
        '_daily_rbc_hvac_kwh',
        denominator_guard_gt=0.05,
        scale=100.0,
    ),
]
daily_ratio_failures = validate_denominator_aware_ratios(
    daily_ratio_frame,
    daily_ratio_contracts,
    table_name='daily_metrics',
)
daily_ratio_contracts_valid = daily_ratio_failures.empty
daily_ratio_contracts_actual = (
    True
    if daily_ratio_contracts_valid
    else format_validation_failures(daily_ratio_failures)
)
pre_analysis_validation = pd.DataFrame([
    _validation_row(
        'required_columns_present', [], missing_required,
        not missing_required,
    ),
    _validation_row(
        'required_numeric_values_finite', True, numeric_finite_actual,
        numeric_finite,
    ),
    _validation_row(
        'denominator_aware_daily_ratio_contracts', True,
        daily_ratio_contracts_actual,
        daily_ratio_contracts_valid,
    ),
])
if not bool(pre_analysis_validation['passed'].all()):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    _write_csv_exclusive(
        pre_analysis_validation, artifact_paths['validation']
    )
    failed_checks = pre_analysis_validation.loc[
        ~pre_analysis_validation['passed'], 'check'
    ].tolist()
    raise RuntimeError(
        f'Missing/non-finite sweep data; validation failed: {failed_checks}'
    )

gap_available = pd.to_numeric(
    decision_diagnostics_df['solver_mip_gap_available'], errors='coerce'
)
gap_values = pd.to_numeric(
    decision_diagnostics_df['solver_mip_gap'], errors='coerce'
)
gap_availability_binary = bool(gap_available.isin([0, 1]).all())
available_gap_values = gap_values.loc[gap_available == 1]
available_gap_valid = bool(
    np.isfinite(available_gap_values.to_numpy(dtype=float)).all()
    and bool((gap_values.loc[gap_available == 1] >= 0.0).all())
)
unavailable_gap_nan = bool(
    gap_values.loc[gap_available == 0].isna().all()
)

solver_recovery_validation = validate_solver_recovery_provenance(
    decision_diagnostics_df, GRB
)
solver_recovery_provenance_valid = solver_recovery_validation[
    'provenance_valid'
]
solver_recovery_runtime_accounting_valid = solver_recovery_validation[
    'runtime_accounting_valid'
]

daily_miqp_metrics = daily_metrics_df.loc[
    daily_metrics_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
]
daily_diagnostic_numeric = [
    'decision_count', 'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s', 'solver_runtime_sum_s',
    'solver_runtime_mean_s', 'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
]
daily_diagnostic_summaries_finite = _finite_numeric(
    daily_miqp_metrics, daily_diagnostic_numeric
)

miqp_timeseries = all_timeseries_df.loc[
    all_timeseries_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
].copy()
rbc_timeseries = all_timeseries_df.loc[
    all_timeseries_df['controller_objective'] == 'baseline_rbc'
]
rbc_hvac_kwh = float(rbc_timeseries['hvac_kw'].sum()) / 60.0
if rbc_hvac_kwh <= 0.05:
    raise RuntimeError(
        f'Pooled RBC HVAC energy is too small for shedding: {rbc_hvac_kwh}'
    )
pooled_records = []
for controller_objective in ('no_pv', 'onsite_pv'):
    for weight in SWEEP_WEIGHTS:
        pooled_case = miqp_timeseries.loc[
            (miqp_timeseries['controller_objective'] == controller_objective)
            & (miqp_timeseries['w_x_slack'] == weight)
        ].copy()
        if pooled_case.empty:
            raise RuntimeError(
                f'Missing pooled rows for {controller_objective}, wx={weight}'
            )
        thermal = temperature_metrics(
            pooled_case, AC_STATE_COLS, bound_c=30.0
        )
        matching_daily = daily_metrics_df.loc[
            (daily_metrics_df['controller_objective'] == controller_objective)
            & (daily_metrics_df['w_x_slack'] == weight)
        ]
        hvac_kwh = float(pooled_case['hvac_kw'].sum()) / 60.0
        pv_available_kwh = float(pooled_case['pv_kw'].sum()) / 60.0
        grid_kwh = float(pooled_case['grid_kw'].sum()) / 60.0
        export_kwh = float(pooled_case['export_kw'].sum()) / 60.0
        self_consumed_kwh = float(pooled_case['self_kw'].sum()) / 60.0
        energy_shedding_vs_rbc_kwh = rbc_hvac_kwh - hvac_kwh
        pooled_peak_windows = peak_window_metrics(pooled_case)
        pooled_diagnostics = decision_diagnostics_df.loc[
            (decision_diagnostics_df['controller_objective'] == controller_objective)
            & (decision_diagnostics_df['w_x_slack'] == weight)
        ]
        pooled_diagnostic_summary = summarize_decision_diagnostics(
            pooled_diagnostics
        )
        pooled_records.append({
            'controller_objective': controller_objective,
            'w_x_slack': float(weight),
            **thermal,
            'post5_any_zone_minutes_above_30': int(
                matching_daily['post5_any_zone_minutes_above_30'].sum()
            ),
            'post5_zone_degree_minutes_above_30': float(
                matching_daily['post5_zone_degree_minutes_above_30'].sum()
            ),
            'hvac_kwh': hvac_kwh,
            'pv_available_kwh': pv_available_kwh,
            'grid_kwh': grid_kwh,
            'export_kwh': export_kwh,
            'self_consumed_kwh': self_consumed_kwh,
            'self_consumption_ratio': (
                self_consumed_kwh / pv_available_kwh
                if pv_available_kwh > 0 else np.nan
            ),
            'self_sufficiency_ratio': (
                self_consumed_kwh / hvac_kwh if hvac_kwh > 0 else np.nan
            ),
            'energy_shedding_vs_rbc_kwh': energy_shedding_vs_rbc_kwh,
            'energy_shedding_vs_rbc_pct': (
                100.0 * energy_shedding_vs_rbc_kwh / rbc_hvac_kwh
            ),
            'peak_hvac_kw': float(pooled_case['hvac_kw'].max()),
            'peak_grid_kw': float(pooled_case['grid_kw'].max()),
            **pooled_peak_windows,
            'pfc_on_fraction': float(
                pooled_case['pfc_on'].astype(int).mean()
            ),
            'window_open_fraction': float(
                (1 - pooled_case['z'].astype(int)).mean()
            ),
            'switch_count': int(matching_daily['switch_count'].sum()),
            **pooled_diagnostic_summary,
        })

pooled_metrics_df = pd.DataFrame(pooled_records).sort_values(
    ['controller_objective', 'w_x_slack'], kind='stable'
).reset_index(drop=True)
pareto_table_df, selection_summary = build_balanced_selection(
    pooled_metrics_df
)

cases_per_date = daily_metrics_df.groupby('date')['case'].nunique()
minutes_per_case_date = all_timeseries_df.groupby(['date', 'case']).size()
decisions_per_case_date = decision_diagnostics_df.groupby(
    ['date', 'case']
).size()
accepted_solver_statuses = {GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT}
solver_status_values = set(
    decision_diagnostics_df['solver_status'].astype(int).unique().tolist()
)
rain_lockout_violations = int(
    daily_metrics_df['rain_safety_violation_minutes'].sum()
)
dwell_lock_violations = int((
    (decision_diagnostics_df['effective_dwell_lock_steps'] > 0)
    & (decision_diagnostics_df['rain_force_ac'] == 0)
    & (
        decision_diagnostics_df['chosen_mode_z']
        != decision_diagnostics_df['previous_mode_z']
    )
).sum())
pfcu_ac_violations = int((
    (decision_diagnostics_df['chosen_mode_z'] == 1)
    & (decision_diagnostics_df['pfcu_on'] != 0)
).sum())

peak_flag_columns = ['morning_peak_available', 'evening_peak_available']
peak_flags = pd.concat(
    [daily_metrics_df[peak_flag_columns], pooled_metrics_df[peak_flag_columns]],
    ignore_index=True,
).apply(pd.to_numeric, errors='coerce')
peak_window_flags_binary = bool(
    peak_flags.isin([0, 1]).to_numpy().all()
)
full_morning_peak_metrics_available = bool(
    window_mode != 'full'
    or (
        (daily_metrics_df['morning_peak_available'] == 1).all()
        and (pooled_metrics_df['morning_peak_available'] == 1).all()
        and _finite_numeric(
            daily_metrics_df,
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw'],
        )
        and _finite_numeric(
            pooled_metrics_df,
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw'],
        )
    )
)
full_evening_peak_metrics_available = bool(
    window_mode != 'full'
    or (
        (daily_metrics_df['evening_peak_available'] == 1).all()
        and (pooled_metrics_df['evening_peak_available'] == 1).all()
        and _finite_numeric(
            daily_metrics_df,
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw'],
        )
        and _finite_numeric(
            pooled_metrics_df,
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw'],
        )
    )
)
smoke_morning_peak_metrics_unavailable = bool(
    window_mode != 'smoke'
    or (
        (daily_metrics_df['morning_peak_available'] == 0).all()
        and (pooled_metrics_df['morning_peak_available'] == 0).all()
        and daily_metrics_df[
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw']
        ].isna().to_numpy().all()
        and pooled_metrics_df[
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw']
        ].isna().to_numpy().all()
    )
)
smoke_evening_peak_metrics_unavailable = bool(
    window_mode != 'smoke'
    or (
        (daily_metrics_df['evening_peak_available'] == 0).all()
        and (pooled_metrics_df['evening_peak_available'] == 0).all()
        and daily_metrics_df[
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw']
        ].isna().to_numpy().all()
        and pooled_metrics_df[
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw']
        ].isna().to_numpy().all()
    )
)
pooled_reporting_numeric = [
    'energy_shedding_vs_rbc_kwh', 'energy_shedding_vs_rbc_pct',
    'morning_peak_observed_minutes', 'morning_peak_expected_minutes',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'pfc_on_fraction', 'decision_count', 'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s', 'solver_runtime_sum_s',
    'solver_runtime_mean_s', 'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
]
pooled_reporting_metrics_finite = _finite_numeric(
    pooled_metrics_df, pooled_reporting_numeric
)

validation_records = [
    _validation_row('no_error_rows', 0, len(errors_df), len(errors_df) == 0),
    _validation_row(
        'cpu_device_selected', 'cpu', str(hooks.device),
        str(hooks.device) == 'cpu',
    ),
    _validation_row(
        'three_sweep_weights', [100.0, 1000.0, 10000.0],
        list(SWEEP_WEIGHTS),
        tuple(SWEEP_WEIGHTS) == (100.0, 1000.0, 10000.0),
    ),
    _validation_row(
        'selected_date_count', expected_date_count,
        len(selected_dates), len(selected_dates) == expected_date_count,
    ),
    _validation_row(
        'daily_row_count', expected_daily_rows,
        len(daily_metrics_df), len(daily_metrics_df) == expected_daily_rows,
    ),
    _validation_row(
        'timeseries_row_count', expected_timeseries_rows,
        len(all_timeseries_df), len(all_timeseries_df) == expected_timeseries_rows,
    ),
    _validation_row(
        'miqp_decision_count', expected_miqp_decisions,
        len(decision_diagnostics_df),
        len(decision_diagnostics_df) == expected_miqp_decisions,
    ),
    _validation_row(
        'daily_cases_per_date', expected_case_count,
        sorted(cases_per_date.unique().tolist()),
        len(cases_per_date) == expected_date_count
        and bool((cases_per_date == expected_case_count).all()),
    ),
    _validation_row(
        'timeseries_rows_per_case_date', expected_minutes_per_case_date,
        sorted(minutes_per_case_date.unique().tolist()),
        len(minutes_per_case_date) == expected_daily_rows
        and bool((minutes_per_case_date == expected_minutes_per_case_date).all()),
    ),
    _validation_row(
        'decision_rows_per_miqp_case_date',
        expected_decisions_per_miqp_case_date,
        sorted(decisions_per_case_date.unique().tolist()),
        len(decisions_per_case_date) == expected_date_count * expected_miqp_case_count
        and bool((decisions_per_case_date == expected_decisions_per_miqp_case_date).all()),
    ),
    _validation_row(
        'required_columns_present', [], missing_required,
        not missing_required,
    ),
    _validation_row(
        'required_numeric_values_finite', True,
        numeric_finite_actual, numeric_finite,
    ),
    _validation_row(
        'denominator_aware_daily_ratio_contracts', True,
        daily_ratio_contracts_actual,
        daily_ratio_contracts_valid,
    ),
    _validation_row(
        'solver_statuses_accepted', sorted(accepted_solver_statuses),
        sorted(solver_status_values),
        solver_status_values.issubset(accepted_solver_statuses),
    ),
    _validation_row(
        'solver_mip_gap_availability_binary', True,
        gap_availability_binary, gap_availability_binary,
    ),
    _validation_row(
        'available_solver_mip_gap_finite_nonnegative', True,
        available_gap_valid, available_gap_valid,
    ),
    _validation_row(
        'unavailable_solver_mip_gap_is_nan', True,
        unavailable_gap_nan, unavailable_gap_nan,
    ),
    _validation_row(
        'solver_recovery_provenance_valid', True,
        solver_recovery_provenance_valid, solver_recovery_provenance_valid,
    ),
    _validation_row(
        'solver_recovery_runtime_accounting_valid', True,
        solver_recovery_runtime_accounting_valid,
        solver_recovery_runtime_accounting_valid,
    ),
    _validation_row(
        'daily_diagnostic_summaries_finite', True,
        daily_diagnostic_summaries_finite,
        daily_diagnostic_summaries_finite,
    ),
    _validation_row(
        'pooled_reporting_metrics_finite', True,
        pooled_reporting_metrics_finite,
        pooled_reporting_metrics_finite,
    ),
    _validation_row(
        'peak_window_availability_flags_binary', True,
        peak_window_flags_binary, peak_window_flags_binary,
    ),
    _validation_row(
        'morning_peak_metrics_available_if_full', True,
        full_morning_peak_metrics_available,
        full_morning_peak_metrics_available,
    ),
    _validation_row(
        'evening_peak_metrics_available_if_full', True,
        full_evening_peak_metrics_available,
        full_evening_peak_metrics_available,
    ),
    _validation_row(
        'smoke_morning_peak_metrics_unavailable', True,
        smoke_morning_peak_metrics_unavailable,
        smoke_morning_peak_metrics_unavailable,
    ),
    _validation_row(
        'smoke_evening_peak_metrics_unavailable', True,
        smoke_evening_peak_metrics_unavailable,
        smoke_evening_peak_metrics_unavailable,
    ),
    _validation_row(
        'rain_lockout_safety', 0, rain_lockout_violations,
        rain_lockout_violations == 0,
    ),
    _validation_row(
        'dwell_lock_safety', 0, dwell_lock_violations,
        dwell_lock_violations == 0,
    ),
    _validation_row(
        'pfcu_disabled_in_ac_mode', 0, pfcu_ac_violations,
        pfcu_ac_violations == 0,
    ),
]
validation_df = pd.DataFrame(validation_records)

if not bool(validation_df['passed'].all()):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    _write_csv_exclusive(validation_df, artifact_paths['validation'])
    failed_checks = validation_df.loc[
        ~validation_df['passed'], 'check'
    ].tolist()
    raise RuntimeError(f'Pre-report validation failed: {failed_checks}')

_write_csv_exclusive(
    all_timeseries_df,
    artifact_paths['timeseries'],
    compression='gzip',
)
_write_csv_exclusive(
    decision_diagnostics_df,
    artifact_paths['decision_diagnostics'],
    compression='gzip',
)
_write_csv_exclusive(daily_metrics_df, artifact_paths['daily_metrics'])
_write_csv_exclusive(pooled_metrics_df, artifact_paths['pooled_metrics'])
_write_csv_exclusive(pareto_table_df, artifact_paths['pareto_table'])
_write_csv_exclusive(errors_df, artifact_paths['errors'])

display_columns = [
    'controller_objective', 'w_x_slack',
    'zone_degree_minutes_above_30', 'any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'self_consumption_ratio',
    'self_sufficiency_ratio', 'energy_shedding_vs_rbc_kwh',
    'energy_shedding_vs_rbc_pct', 'morning_peak_available',
    'morning_peak_observed_minutes', 'morning_peak_expected_minutes',
    'morning_peak_hvac_kw', 'morning_peak_grid_kw',
    'evening_peak_available', 'evening_peak_observed_minutes',
    'evening_peak_expected_minutes',
    'evening_peak_hvac_kw', 'evening_peak_grid_kw', 'pfc_on_fraction',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction', 'solver_runtime_sum_s',
    'solver_mip_gap_available_fraction', 'solver_mip_gap_mean',
    'window_open_fraction', 'switch_count',
]
pooled_metrics_df[display_columns].round(4)


In [ ]:
# Weight response across the two controller objectives.
controller_colors = {'no_pv': 'tab:blue', 'onsite_pv': 'tab:orange'}
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
response_specs = [
    ('any_zone_minutes_above_30', 'Any-zone violation minutes'),
    ('zone_degree_minutes_above_30', 'Zone degree-minutes above 30 C'),
    ('max_zone_temp_c', 'Maximum zone temperature (C)'),
    ('hvac_kwh', 'HVAC energy (kWh)'),
    ('window_open_fraction', 'Window-open fraction'),
    ('switch_count', 'Mode switches'),
]
for axis, (metric, title) in zip(axes.ravel(), response_specs):
    for objective in ('no_pv', 'onsite_pv'):
        data = pooled_metrics_df.loc[
            pooled_metrics_df['controller_objective'] == objective
        ].sort_values('w_x_slack')
        axis.plot(
            data['w_x_slack'], data[metric], marker='o',
            color=controller_colors[objective], label=objective,
        )
        for _, row in data.iterrows():
            axis.annotate(
                f"{int(row['w_x_slack'])}",
                (row['w_x_slack'], row[metric]),
                xytext=(4, 4), textcoords='offset points', fontsize=8,
            )
    axis.set_xscale('log')
    axis.set_title(title)
    axis.set_xlabel('w_x_slack')
    axis.grid(True, alpha=0.25)
    axis.legend()

ratio_axis = axes.ravel()[6]
pv_response = pooled_metrics_df.loc[
    pooled_metrics_df['controller_objective'] == 'onsite_pv'
].sort_values('w_x_slack')
for metric, label, marker in (
    ('self_consumption_ratio', 'SC', 'o'),
    ('self_sufficiency_ratio', 'SS', 's'),
):
    ratio_axis.plot(
        pv_response['w_x_slack'], pv_response[metric],
        marker=marker, label=label,
    )
    for _, row in pv_response.iterrows():
        ratio_axis.annotate(
            f"{int(row['w_x_slack'])}",
            (row['w_x_slack'], row[metric]),
            xytext=(4, -10), textcoords='offset points', fontsize=8,
        )
ratio_axis.set_xscale('log')
ratio_axis.set_ylim(0, 1.08)
ratio_axis.set_title(
    'MPC-PV self-consumption / self-sufficiency', pad=12
)
ratio_axis.set_xlabel('w_x_slack')
ratio_axis.grid(True, alpha=0.25)
ratio_axis.legend()
axes.ravel()[7].axis('off')
fig.suptitle('Weight response', y=1.01)
fig.tight_layout()
_savefig_exclusive(fig, artifact_paths['weight_response'])
plt.close(fig)

# Three-panel Pareto view: ordinary MPC, MPC-PV, and combined normalized.
def _annotate_pareto_weights(axis, data, x_col, y_col):
    grouped = data.groupby([x_col, y_col], sort=False, dropna=False)
    for (x_value, y_value), points in grouped:
        weights = sorted(
            int(weight) for weight in points['w_x_slack'].tolist()
        )
        label = ', '.join(str(weight) for weight in weights)
        if len(weights) > 1:
            label += ' (tied)'
        axis.annotate(
            label,
            (x_value, y_value),
            xytext=(4, 4), textcoords='offset points', fontsize=8,
        )


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, objective, title in zip(
    axes[:2],
    ('no_pv', 'onsite_pv'),
    ('Ordinary MPC', 'MPC-PV'),
):
    data = pareto_table_df.loc[
        pareto_table_df['controller_objective'] == objective
    ].sort_values('w_x_slack')
    frontier = data.loc[data['controller_pareto']]
    axis.scatter(
        data['thermal_pareto_display'], data['energy_pareto_display'],
        color=controller_colors[objective], alpha=0.45,
    )
    axis.plot(
        frontier['thermal_pareto_display'],
        frontier['energy_pareto_display'],
        color=controller_colors[objective], marker='o',
    )
    _annotate_pareto_weights(
        axis, data, 'thermal_pareto_display', 'energy_pareto_display'
    )
    knee = selection_summary['individual'][objective]['knee_weight']
    if knee is not None:
        marked = data.loc[data['w_x_slack'] == knee].iloc[0]
        axis.scatter(
            [marked['thermal_pareto_display']],
            [marked['energy_pareto_display']],
            marker='*', s=180, color='black', label=f'knee={int(knee)}',
        )
        axis.legend()
    if selection_summary['individual'][objective]['no_material_response']:
        axis.text(
            0.03, 0.95, 'No material response within tolerances',
            transform=axis.transAxes, va='top', fontsize=9,
        )
    axis.set_title(title)
    axis.set_xlabel('Zone degree-minutes above 30 C')
    axis.set_ylabel('HVAC energy (kWh)')
    axis.grid(True, alpha=0.25)

combined = (
    pareto_table_df[
        ['w_x_slack', 'combined_thermal_normalized',
         'combined_energy_normalized', 'combined_pareto']
    ]
    .drop_duplicates('w_x_slack')
    .sort_values('w_x_slack')
)
combined_frontier = combined.loc[combined['combined_pareto']]
axes[2].scatter(
    combined['combined_thermal_normalized'],
    combined['combined_energy_normalized'],
    color='tab:green', alpha=0.45,
)
axes[2].plot(
    combined_frontier['combined_thermal_normalized'],
    combined_frontier['combined_energy_normalized'],
    color='tab:green', marker='o',
)
_annotate_pareto_weights(
    axes[2], combined,
    'combined_thermal_normalized', 'combined_energy_normalized',
)
common_knee = selection_summary['common_knee_weight']
if common_knee is not None:
    marked = combined.loc[combined['w_x_slack'] == common_knee].iloc[0]
    axes[2].scatter(
        [marked['combined_thermal_normalized']],
        [marked['combined_energy_normalized']],
        marker='*', s=180, color='black',
        label=f'common knee={int(common_knee)}',
    )
    axes[2].legend()
axes[2].set_title('Combined normalized')
axes[2].set_xlabel('Normalized thermal burden')
axes[2].set_ylabel('Normalized HVAC energy')
axes[2].grid(True, alpha=0.25)
fig.suptitle('Three-panel Pareto', y=1.02)
fig.tight_layout()
_savefig_exclusive(fig, artifact_paths['pareto'])
plt.close(fig)


## Representative days and portable run metadata


In [ ]:
weight_100_daily = daily_metrics_df.loc[
    daily_metrics_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
    & (daily_metrics_df['w_x_slack'] == SWEEP_WEIGHTS[0])
]
hottest_dates = (
    weight_100_daily.groupby('date')['zone_degree_minutes_above_30']
    .max()
    .nlargest(min(3, expected_date_count))
    .index.tolist()
)
representative_records = [
    {
        'date': date,
        'controller_objective': 'both',
        'reason': 'hottest wx=100 day',
        'mode_changes_vs_wx_100': np.nan,
        'selected_weight': selection_summary['common_knee_weight'],
    }
    for date in hottest_dates
]

common_knee = selection_summary['common_knee_weight']
if common_knee is not None:
    for date in selected_dates:
        for objective in ('no_pv', 'onsite_pv'):
            base = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == SWEEP_WEIGHTS[0]),
                ['ts', 'z'],
            ].sort_values('ts')
            selected = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == common_knee),
                ['ts', 'z'],
            ].sort_values('ts')
            aligned = base.merge(
                selected, on='ts', suffixes=('_wx100', '_selected'),
                validate='one_to_one',
            )
            if len(aligned) != expected_minutes_per_case_date:
                raise RuntimeError(
                    f'Representative-day mode alignment failed for {date}, '
                    f'{objective}: {len(aligned)} rows'
                )
            mode_changes = int(
                (aligned['z_wx100'] != aligned['z_selected']).sum()
            )
            if mode_changes >= 35:
                representative_records.append({
                    'date': date,
                    'controller_objective': objective,
                    'reason': 'mode changes vs wx=100 >= 35 minutes',
                    'mode_changes_vs_wx_100': mode_changes,
                    'selected_weight': common_knee,
                })
else:
    for record in representative_records:
        record['reason'] += '; no common knee / selected-weight comparison'

representative_days_df = pd.DataFrame(representative_records).sort_values(
    ['date', 'controller_objective', 'reason'], kind='stable'
).reset_index(drop=True)
representative_dates = representative_days_df['date'].drop_duplicates().tolist()
representative_dir.mkdir(parents=False, exist_ok=False)
_write_csv_exclusive(
    representative_days_df,
    artifact_paths['representative_days'],
)

for date in representative_dates:
    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
    for pair_index, objective in enumerate(('no_pv', 'onsite_pv')):
        baseline = miqp_timeseries.loc[
            (miqp_timeseries['date'] == date)
            & (miqp_timeseries['controller_objective'] == objective)
            & (miqp_timeseries['w_x_slack'] == SWEEP_WEIGHTS[0])
        ].sort_values('ts')
        if baseline.empty:
            raise RuntimeError(
                f'Missing wx=100 representative rows for {date}, {objective}'
            )
        baseline_ts = pd.to_datetime(baseline['ts'])
        baseline_max = baseline[AC_STATE_COLS].max(axis=1)
        temp_axis = axes[pair_index * 2]
        mode_axis = axes[pair_index * 2 + 1]
        temp_axis.plot(
            baseline_ts, baseline_max,
            label=f'{objective} wx=100 max-zone T',
            color=controller_colors[objective], linestyle='--',
        )
        mode_axis.step(
            baseline_ts, baseline['z'], where='post',
            label=f'{objective} wx=100 mode',
            color=controller_colors[objective], linestyle='--',
        )
        mode_axis.step(
            baseline_ts, baseline['pfc_on'], where='post',
            label=f'{objective} wx=100 PFCU',
            color='tab:gray', linestyle=':',
        )
        if common_knee is not None:
            selected = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == common_knee)
            ].sort_values('ts')
            if selected.empty:
                raise RuntimeError(
                    f'Missing selected representative rows for '
                    f'{date}, {objective}, wx={common_knee}'
                )
            selected_ts = pd.to_datetime(selected['ts'])
            selected_max = selected[AC_STATE_COLS].max(axis=1)
            temp_axis.plot(
                selected_ts, selected_max,
                label=f'{objective} wx={int(common_knee)} max-zone T',
                color=controller_colors[objective],
            )
            mode_axis.step(
                selected_ts, selected['z'], where='post',
                label=f'{objective} wx={int(common_knee)} mode',
                color=controller_colors[objective],
            )
            mode_axis.step(
                selected_ts, selected['pfc_on'], where='post',
                label=f'{objective} wx={int(common_knee)} PFCU',
                color='black', linestyle=':',
            )
        temp_axis.axhline(30.0, color='red', linestyle=':', label='30 C bound')
        temp_axis.set_ylabel('Max zone T (C)')
        temp_axis.grid(True, alpha=0.25)
        temp_axis.legend(loc='best', ncol=2)
        mode_axis.set_ylabel('Mode / PFCU')
        mode_axis.set_yticks([0, 1])
        mode_axis.grid(True, alpha=0.25)
        mode_axis.legend(loc='best', ncol=2)
    axes[-1].set_xlabel('Time')
    fig.suptitle(f'Representative day {date}', y=1.01)
    fig.tight_layout()
    _savefig_exclusive(fig, representative_dir / f'{date}.png')
    plt.close(fig)


validation_passed = bool(validation_df["passed"].all())
run_metadata = {
    "study": "Temperature upper-slack penalty sensitivity",
    "input_paths": {
        "observed_data": L14_DATA_PATH.name,
        "ac_model": str(AC_MODEL_PATH.relative_to(REPO_ROOT)),
        "nv_model": str(NV_MODEL_PATH.relative_to(REPO_ROOT)),
        "pv_coefficients": str(PV_COEFFICIENT_PATH.relative_to(REPO_ROOT)),
    },
    "copied_input_manifest": {
        "l14_merged_data_with_rain.csv": {
            "copy": L14_DATA_PATH.name
        }
    },
    "runtime": {
        "device": str(hooks.device),
        "cuda_available": bool(torch.cuda.is_available()),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "varied_configuration": {"parameter": "w_x_slack", "weights": list(SWEEP_WEIGHTS)},
    "fixed_configuration": FIXED_MPC_KWARGS,
    "selected_dates": selected_dates,
    "window_mode": window_mode,
    "validation_passed": validation_passed,
}
artifact_paths["run_metadata"].write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
_write_csv_exclusive(validation_df, artifact_paths["validation"])
print({
    "balanced_common_weight": selection_summary["common_knee_weight"],
    "validation_passed": validation_passed,
    "output_dir": OUTPUT_DIR.name,
})


In [ ]:
failed_validation_checks = validation_df.loc[
    ~validation_df['passed'], 'check'
].tolist()
if failed_validation_checks:
    raise RuntimeError(
        f'Final sweep validation failed: {failed_validation_checks}'
    )

print({
    'balanced_common_weight': selection_summary['common_knee_weight'],
    'pv_metric_warning': selection_summary['pv_metric_warning'],
    'validation_passed': True,
    'output_dir': str(OUTPUT_DIR),
})
validation_df
